# 07 · PPLX Embed v1 0.6B: piloto e índice del catálogo


## 1. Entradas y perfil fijo

In [1]:
import csv
import hashlib
import json
import sys
import platform
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

VERSION = "PPLX_EMBED_V1_06B_DENSO_RAG_V1"
REVISION_CODIGO = "PPLX_EMBED_V1_06B_IMPL_01"
MODELO = "perplexity-ai/pplx-embed-v1-0.6b"
REVISION = "2c4d510dd4a732063c31a0f70193e35067b51fd8"
LOTE = 4
DIMENSION = 1024
LIMITE_TOKENS = 32768
NOMBRE_PROMPT = None
NOMBRE_PROMPT_DOCUMENTO = None
PREFIJO = ""
PREFIJO_DOCUMENTO = ""
ATENCION = "sdpa"  # Perfil PyTorch local: no requiere instalar FlashAttention.
CUANTIZACION_NATIVA = "int8"
# Se permite solo el código Python del repositorio oficial, fijado a REVISION.
CONFIAR_CODIGO_OFICIAL = True
SHA_ESPERADO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
# Precaución operativa: pausa después de una llamada lenta, NO criterio de calidad ni timeout CUDA.
PAUSA_TRAS_LLAMADA_S = 30.0


def sha(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


def ahora():
    return datetime.now(timezone.utc).isoformat()


REPO = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta desde 07_RAG o desde la raíz del repositorio.")
BASE = REPO / "07_RAG"
CATALOGO = REPO / "06_grafos/catalogo_articulos.csv"
AUDITORIA = BASE / "auditoria_catalogo.json"
CONSULTAS = BASE / "consultas_para_revision.csv"
VECTORES = BASE / "pplx_0_6b_vectores_v1.npz"
INDICE = BASE / "pplx_0_6b_indice_v1.faiss"

SHA_CATALOGO = sha(CATALOGO)
SHA_AUDITORIA = sha(AUDITORIA)
auditoria = json.loads(AUDITORIA.read_text(encoding="utf-8"))
if SHA_CATALOGO != SHA_ESPERADO or SHA_CATALOGO != auditoria["fuentes"]["catalogo"]["sha256_bytes"]:
    raise ValueError("El catálogo no coincide con la versión auditada. No continuar.")
if auditoria["resumen"]["incidencias_bloqueantes"] != 0:
    raise ValueError("Hay incidencias bloqueantes en la auditoría.")
with CATALOGO.open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))
article_ids = [r["article_id"] for r in articulos]
resumenes = [r["abstract"] for r in articulos]
if len(article_ids) != 519 or len(set(article_ids)) != 519 or len(set(resumenes)) != 514:
    raise ValueError("Cambió la estructura del catálogo.")
if not all(t.strip() for t in resumenes):
    raise ValueError("Hay resúmenes vacíos; no se sustituyen con texto generado.")
with CONSULTAS.open(encoding="utf-8-sig", newline="") as f:
    consultas = list(csv.DictReader(f))
por_query = {r["query_id"]: r for r in consultas}
if len(por_query) != len(consultas) or not {"N01-ES", "N01-EN"}.issubset(por_query):
    raise ValueError("Revisar identificadores de las consultas; no se inventan consultas sustitutas.")
SHA_CONSULTAS = sha(CONSULTAS)

PERFIL = {
    "version": VERSION, "modelo": MODELO, "revision": REVISION,
    "dispositivo": "cuda", "dtype_pesos": "float32", "dimensiones": DIMENSION,
    "batch_size": LOTE, "limite_tokens": LIMITE_TOKENS, "campo": "abstract",
    "prefijo_documento": PREFIJO_DOCUMENTO, "nombre_prompt_documento": NOMBRE_PROMPT_DOCUMENTO,
    "nombre_prompt_consulta": NOMBRE_PROMPT,
    "prefijo_consulta": PREFIJO, "pooling": "mean", "include_prompt": True,
    "atencion_solicitada": ATENCION, "arquitectura_esperada": "bidirectional_pplx_qwen3",
    "trust_remote_code": CONFIAR_CODIGO_OFICIAL, "code_revision": REVISION,
    "cuantizacion_nativa_salida": CUANTIZACION_NATIVA,
    "precision_encode_ST": "float32",
    "normalizar_en_encode": False,
    "recodificacion_int8_ST": False,
    "torch_compile_invocado": False,
    "license_model_card": "MIT",
    "normalizacion": "L2 float32 sobre la salida cuantizada nativa; no deshace la cuantizacion", "indice": "IndexFlatIP CPU",
    "orden_lotes": "consecutivos en el orden del catalogo; orden interno de Sentence Transformers",
    "pausa_operativa_s": PAUSA_TRAS_LLAMADA_S,
}
FIRMA = hashlib.sha256(json.dumps(
    {"perfil": PERFIL, "catalogo": SHA_CATALOGO}, sort_keys=True,
    ensure_ascii=False).encode("utf-8")).hexdigest()

print("Protocolo:", VERSION, "| Implementación:", REVISION_CODIGO)
print("Modelo:", MODELO, "| Revisión:", REVISION)
print("Catálogo verificado:", SHA_CATALOGO)
print("Artículos:", len(article_ids), "| Resúmenes distintos:", len(set(resumenes)))
print("Consultas:", len(consultas), "| Revisión humana:",
      dict(Counter(r.get("revision_humana", "no registrada") for r in consultas)))
print("Salida:", BASE)
print("Esta celda NO carga modelos ni crea archivos.")

Protocolo: PPLX_EMBED_V1_06B_DENSO_RAG_V1 | Implementación: PPLX_EMBED_V1_06B_IMPL_01
Modelo: perplexity-ai/pplx-embed-v1-0.6b | Revisión: 2c4d510dd4a732063c31a0f70193e35067b51fd8
Catálogo verificado: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Artículos: 519 | Resúmenes distintos: 514
Consultas: 24 | Revisión humana: {'pendiente': 24}
Salida: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Esta celda NO carga modelos ni crea archivos.


## 2. Bibliotecas y funciones

In [2]:
import numpy as np
import torch
import faiss
import sentence_transformers
import transformers
import huggingface_hub
import inspect
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

ENTORNO = {
    "python": sys.version.split()[0], "sistema": platform.platform(),
    "numpy": np.__version__, "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "sentence_transformers": sentence_transformers.__version__,
    "transformers": transformers.__version__, "huggingface_hub": huggingface_hub.__version__,
    "faiss": getattr(faiss, "__version__", "no_registrada"),
}
if not torch.cuda.is_available():
    raise RuntimeError("Este perfil requiere CUDA. No se cambia a CPU automáticamente.")
ENTORNO["gpu"] = torch.cuda.get_device_name(0)
for nombre, valor in ENTORNO.items():
    print(nombre + ":", valor)
libre, total = torch.cuda.mem_get_info()
print("VRAM libre / total (GiB):", round(libre / 2**30, 2), "/", round(total / 2**30, 2))


def comprobar_entradas():
    if sha(CATALOGO) != SHA_CATALOGO or sha(AUDITORIA) != SHA_AUDITORIA:
        raise ValueError("Cambió una entrada durante la sesión. No continuar.")
    if sha(CONSULTAS) != SHA_CONSULTAS:
        raise ValueError("Cambió el CSV de consultas; vuelve a ejecutar la celda 1.")


def memoria():
    libre, total = torch.cuda.mem_get_info()
    return {
        "libre_bytes": int(libre), "total_bytes": int(total),
        "asignada_pytorch_bytes": int(torch.cuda.memory_allocated()),
        "reservada_pytorch_bytes": int(torch.cuda.memory_reserved()),
        "pico_asignada_pytorch_bytes": int(torch.cuda.max_memory_allocated()),
        "pico_reservada_pytorch_bytes": int(torch.cuda.max_memory_reserved()),
        "alcance": "PyTorch del proceso y lecturas puntuales CUDA; no mide RAM del sistema",
    }


def cargar_estado():
    if not VECTORES.exists():
        if INDICE.exists():
            raise FileExistsError("Existe un índice sin NPZ asociado; revisar sin borrar nada.")
        meta = {
            "firma": FIRMA, "perfil": PERFIL, "creado_utc": ahora(), "estado": "nuevo",
            "sha256_catalogo": SHA_CATALOGO, "sha256_auditoria_inicial": SHA_AUDITORIA,
            "pilotos": [], "cargas": [], "sesiones_catalogo": [], "lotes": [],
            "errores": [], "busquedas": [], "entorno_vectorizacion": None,
        }
        return meta, np.empty((0, DIMENSION), dtype=np.float32)
    with np.load(VECTORES, allow_pickle=False) as z:
        meta = json.loads(z["metadata_json"].item())
        matriz = z["vectores"].copy()
        ids = z["article_ids"].tolist()
        if (meta["firma"] != FIRMA or meta["perfil"] != PERFIL
                or z["article_ids_catalogo"].tolist() != article_ids
                or z["modelo"].item() != MODELO or z["modelo_revision"].item() != REVISION
                or z["sha256_catalogo"].item() != SHA_CATALOGO):
            raise ValueError("El NPZ pertenece a otro perfil, catálogo o versión. No se sobrescribe.")
        if z["sha256_matriz"].item() != hashlib.sha256(matriz.tobytes()).hexdigest():
            raise ValueError("La matriz guardada no coincide con su huella.")
    if (matriz.dtype != np.float32 or matriz.ndim != 2 or matriz.shape[1] != DIMENSION
            or len(matriz) > len(article_ids) or ids != article_ids[:len(matriz)]):
        raise ValueError("Forma u orden de las filas del NPZ incorrectos.")
    if not np.isfinite(matriz).all():
        raise ValueError("Hay valores no finitos en el NPZ.")
    if len(matriz) and not np.allclose(np.linalg.norm(matriz, axis=1), 1, rtol=0, atol=1e-5):
        raise ValueError("Las normas guardadas no son unitarias.")
    cursor = 0
    for registro in meta["lotes"]:
        fin = registro["fin_fila_exclusivo"]
        if (registro["inicio_fila"] != cursor or not cursor < fin <= len(matriz)
                or registro["article_ids"] != article_ids[cursor:fin]):
            raise ValueError("El historial de lotes no corresponde a la matriz guardada.")
        cursor = fin
    if cursor != len(matriz):
        raise ValueError("Hay vectores sin lote registrado o lotes repetidos.")
    if meta["estado"] == "completo" and len(matriz) != len(article_ids):
        raise ValueError("Un checkpoint incompleto no puede declararse completo.")
    if INDICE.exists() and len(matriz) != len(article_ids):
        raise ValueError("Un índice existe junto a un NPZ incompleto. Revisar sin sobrescribir.")
    return meta, matriz


def guardar_estado(meta, matriz):
    # Sustituye solo el avance de ESTE perfil. Los resultados de otros modelos no se tocan.
    meta["actualizado_utc"] = ahora()
    temporal = VECTORES.with_name(VECTORES.name + ".tmp")
    if temporal.exists():
        raise FileExistsError("Hay un .tmp de una interrupción de escritura; conservarlo y revisar.")
    try:
        with temporal.open("xb") as f:
            np.savez_compressed(
                f, vectores=matriz, article_ids=np.asarray(article_ids[:len(matriz)]),
                article_ids_catalogo=np.asarray(article_ids),
                modelo=np.asarray(MODELO), modelo_revision=np.asarray(REVISION),
                sha256_catalogo=np.asarray(SHA_CATALOGO), campo=np.asarray("abstract"),
                completo=np.asarray(meta["estado"] == "completo"), estado=np.asarray(meta["estado"]),
                sha256_matriz=np.asarray(hashlib.sha256(matriz.tobytes()).hexdigest()),
                tiempo_vectorizacion_s=np.asarray(sum(r["encode_s"] for r in meta["lotes"])),
                tokens_por_articulo=np.asarray(meta.get("modelo_runtime", {}).get("tokens_por_articulo", []), dtype=np.int64),
                metadata_json=np.asarray(json.dumps(meta, ensure_ascii=False, allow_nan=False)),
            )
        temporal.replace(VECTORES)
    except BaseException:
        print("Escritura interrumpida. Se conserva el NPZ anterior y, si existe, su .tmp para revisión.")
        raise


def registrar_error(meta, matriz, etapa, error):
    meta["errores"].append({"fecha_utc": ahora(), "etapa": etapa,
                            "tipo": type(error).__name__, "mensaje": str(error),
                            "revision_codigo": REVISION_CODIGO})
    if meta["estado"] != "completo":
        meta["estado"] = "interrumpido_o_error"
    guardar_estado(meta, matriz)


def verificar_pooling_compatible(modelo):
    """Comprobar el módulo cargado sin cambiar sus parámetros ni su forward."""
    pooling = modelo[1]  # La revisión fijada de modules.json coloca Pooling en el índice 1.
    if type(pooling).__name__ != "Pooling":
        raise ValueError("El módulo 1 no es Pooling; no se fuerza otra arquitectura.")
    obtener_config = getattr(pooling, "get_config_dict", None)
    if not callable(obtener_config):
        raise ValueError("No se puede consultar la configuración efectiva de Pooling.")
    config_pooling = obtener_config()
    if not isinstance(config_pooling, dict):
        raise ValueError("La configuración efectiva de Pooling no es un diccionario.")

    # API actual: un modo como cadena, o varios modos como lista/tupla.
    # Los modos adicionales no son admisibles para esta revisión de PPLX.
    banderas_legacy = {
        "pooling_mode_cls_token": "cls",
        "pooling_mode_max_tokens": "max",
        "pooling_mode_mean_tokens": "mean",
        "pooling_mode_mean_sqrt_len_tokens": "mean_sqrt_len_tokens",
        "pooling_mode_weightedmean_tokens": "weightedmean",
        "pooling_mode_lasttoken": "lasttoken",
    }
    modo = config_pooling.get("pooling_mode")
    if isinstance(modo, str):
        modos = (modo,)
        formato = "pooling_mode"
    elif isinstance(modo, (list, tuple)) and all(isinstance(x, str) for x in modo):
        modos = tuple(modo)
        formato = "pooling_mode"
    elif modo is None and any(k in config_pooling for k in banderas_legacy):
        if any(config_pooling[k] not in (True, False) for k in banderas_legacy if k in config_pooling):
            raise ValueError("Las banderas antiguas de Pooling no son booleanas.")
        modos = tuple(v for k, v in banderas_legacy.items() if config_pooling.get(k, False))
        formato = "banderas_legacy"
    else:
        raise ValueError(f"Formato de Pooling no reconocido: {config_pooling!r}")

    # No admitir declaraciones contradictorias si una implementación expone ambos formatos.
    if formato == "pooling_mode":
        for clave, nombre in banderas_legacy.items():
            if clave in config_pooling and config_pooling[clave] != (nombre in modos):
                raise ValueError("La configuración mezcla declaraciones de Pooling contradictorias.")
    if modos != ("mean",):
        raise ValueError(f"Pooling realmente distinto del esperado: {modos!r}. No se modifica.")
    if config_pooling.get("include_prompt") is not True:
        raise ValueError("PPLX debe conservar include_prompt=True en su módulo Pooling.")
    dimension_pooling = config_pooling.get(
        "embedding_dimension", config_pooling.get("word_embedding_dimension")
    )
    if dimension_pooling != DIMENSION:
        raise ValueError(f"Dimensión de Pooling inesperada: {dimension_pooling!r}.")
    return {
        "modos": list(modos), "formato": formato,
        "clase": f"{type(pooling).__module__}.{type(pooling).__name__}",
        "configuracion": json.loads(json.dumps(config_pooling, ensure_ascii=False, allow_nan=False)),
    }


def registrar_fuentes_oficiales():
    """Descarga a la caché, lee y registra archivos; no ejecuta código aquí."""
    nombres = ["config.json", "modules.json", "1_Pooling/config.json",
               "configuration.py", "modeling.py", "st_quantize.py", "tokenizer_config.json"]
    procedencia, contenidos = {}, {}
    for nombre in nombres:
        ruta = Path(hf_hub_download(repo_id=MODELO, filename=nombre, revision=REVISION))
        datos = ruta.read_bytes()
        procedencia[nombre] = {"revision": REVISION, "sha256": hashlib.sha256(datos).hexdigest(),
                               "bytes": len(datos)}
        if nombre.endswith(".json"):
            contenidos[nombre] = json.loads(datos.decode("utf-8"))
    modulos = contenidos["modules.json"]
    if [x["type"] for x in modulos] != [
        "sentence_transformers.models.Transformer", "sentence_transformers.models.Pooling",
        "st_quantize.FlexibleQuantizer"
    ] or "quantization" not in modulos[2].get("kwargs", []):
        raise ValueError("La revisión no tiene Transformer + Pooling + FlexibleQuantizer esperados.")
    conf = contenidos["config.json"]
    if (conf.get("model_type") != "bidirectional_pplx_qwen3"
            or conf.get("hidden_size") != DIMENSION
            or conf.get("max_position_embeddings") != LIMITE_TOKENS
            or conf.get("dtype", conf.get("torch_dtype")) != "float32"):
        raise ValueError("La configuración oficial no corresponde al perfil fijo.")
    if conf.get("auto_map") != {"AutoConfig": "configuration.PPLXQwen3Config",
                                "AutoModel": "modeling.PPLXQwen3Model"}:
        raise ValueError("AutoConfig/AutoModel no corresponden al código local del mismo repositorio.")
    return procedencia


def asegurar_modelo(meta):
    global codificador_pplx, runtime_pplx, carga_pplx, firma_cargada
    if "codificador_pplx" not in globals():
        if not CONFIAR_CODIGO_OFICIAL:
            raise RuntimeError("PPLX necesita su código oficial. No se sustituye por un modelo genérico.")
        print("Cargando PPLX desde una revisión fija; se ejecutará su código Python oficial.", flush=True)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        inicio = perf_counter()
        fuentes = registrar_fuentes_oficiales()
        m = SentenceTransformer(
            MODELO, revision=REVISION, device="cuda", trust_remote_code=True,
            model_kwargs={"dtype": torch.float32, "attn_implementation": ATENCION,
                          "code_revision": REVISION},
            config_kwargs={"code_revision": REVISION},
        )
        m.eval()
        torch.cuda.synchronize()
        carga_s = perf_counter() - inicio
        if any(p.dtype != torch.float32 or p.device.type != "cuda" for p in m.parameters()):
            raise ValueError("Los pesos no están íntegramente en CUDA/float32; no se cambia el perfil.")
        obtener_dimension = getattr(m, "get_embedding_dimension", None)
        if not callable(obtener_dimension):
            obtener_dimension = getattr(m, "get_sentence_embedding_dimension", None)
        if not callable(obtener_dimension) or obtener_dimension() != DIMENSION:
            raise ValueError("Dimensión distinta de la documentada.")
        prompts = getattr(m, "prompts", {}) or {}
        if (getattr(m, "default_prompt_name", None) is not None
                or any(texto not in (None, "") for texto in prompts.values())):
            raise ValueError("Prefijos inesperados: PPLX no requiere instrucciones.")
        if len(m) != 3 or type(m[2]).__name__ != "FlexibleQuantizer":
            raise ValueError("Se esperaba la cuantización nativa, no Normalize ni otra variante.")
        cuantizador = getattr(m[2], "_int8_quantizer", None)
        if (cuantizador is None or getattr(cuantizador, "_hard", None) is not True
                or getattr(cuantizador, "qmin", None) != -128
                or getattr(cuantizador, "qmax", None) != 127):
            raise ValueError("No coincide el cuantizador nativo int8 de la revisión fijada.")
        conf = m[0].auto_model.config
        if (getattr(conf, "model_type", None) != "bidirectional_pplx_qwen3"
                or getattr(conf, "hidden_size", None) != DIMENSION):
            raise ValueError("La arquitectura no corresponde a PPLX Qwen3 bidireccional.")
        if getattr(m, "truncate_dim", None) is not None:
            raise ValueError("No se permite recortar la dimensión de salida.")
        if getattr(m[0], "do_lower_case", False):
            raise ValueError("No se espera conversión a minúsculas de los documentos.")
        if getattr(conf, "_attn_implementation", None) != ATENCION:
            raise ValueError("No se cargó la implementación SDPA solicitada.")
        capas = getattr(m[0].auto_model, "layers", None)
        if capas is None or any(getattr(c.self_attn, "is_causal", None) is not False for c in capas):
            raise ValueError("No se confirmó atención bidireccional en las capas cargadas.")
        # Registrar y contrastar los bytes del código realmente importado; no modificarlo.
        codigo_cargado = {}
        for nombre, clase in [("configuration.py", type(conf)),
                              ("modeling.py", type(m[0].auto_model)),
                              ("st_quantize.py", type(m[2]))]:
            ruta_codigo = Path(inspect.getfile(clase))
            h = sha(ruta_codigo)
            if h != fuentes[nombre]["sha256"]:
                raise ValueError(f"El código cargado {nombre} difiere del de la revisión fijada.")
            codigo_cargado[nombre] = {"sha256": h, "clase": f"{clase.__module__}.{clase.__name__}"}
        verificacion_pooling = verificar_pooling_compatible(m)
        print("Pooling verificado:", verificacion_pooling["modos"],
              "| include_prompt:", verificacion_pooling["configuracion"]["include_prompt"], flush=True)
        original = int(m.max_seq_length)
        if original <= 0:
            raise ValueError("Límite de entrada inválido.")
        m.max_seq_length = min(original, LIMITE_TOKENS)
        tokenizados = m.tokenizer(resumenes, add_special_tokens=True, padding=False, truncation=False)
        longitudes = [len(x) for x in tokenizados["input_ids"]]
        del tokenizados
        excedidos = [article_ids[i] for i,n in enumerate(longitudes) if n > m.max_seq_length]
        if excedidos:
            raise ValueError("Entradas demasiado largas; no se truncan: " + ", ".join(excedidos))
        commit = getattr(conf, "_commit_hash", None)
        if commit is not None and commit != REVISION:
            raise ValueError("La revisión reportada por Transformers difiere de la solicitada.")
        runtime_pplx = {
            "dtype_pesos": str(next(m.parameters()).dtype), "revision_solicitada": REVISION,
            "revision_reportada": commit, "code_revision": REVISION, "trust_remote_code": True,
            "dimension": DIMENSION, "max_seq_length_original": original,
            "max_seq_length": int(m.max_seq_length), "tokens_por_articulo": longitudes,
            "articulos_truncados": 0, "prefijo_consulta": "", "prefijo_documento": "",
            "prompts_reportados": prompts, "arquitectura": conf.model_type,
            "precision_vectores_guardados": "float32 L2 de la representación cuantizada nativa",
            "cuantizacion_nativa_salida": CUANTIZACION_NATIVA,
            "precision_encode_ST": "float32 (sin recuantizacion posterior de ST)",
            "tokenizacion": "texto original sin instrucciones; tokens especiales del modelo; sin truncamiento",
            "modulos": [type(modulo).__name__ for modulo in m],
            "atencion_reportada": getattr(conf, "_attn_implementation", None),
            "padding_side": m.tokenizer.padding_side, "pooling_verificado": verificacion_pooling,
            "fuentes_oficiales": fuentes, "codigo_importado": codigo_cargado,
            "torch_compile_invocado": False,
        }
        carga_pplx = {"id": str(uuid4()), "fecha_utc": ahora(), "tiempo_carga_s": carga_s,
                      "revision_codigo": REVISION_CODIGO, "entorno": ENTORNO,
                      "memoria_despues": memoria()}
        codificador_pplx, firma_cargada = m, FIRMA
    elif firma_cargada != FIRMA:
        raise ValueError("Hay otro perfil cargado. Reinicia el kernel antes de cambiarlo.")
    anterior = meta.get("modelo_runtime")
    if anterior is not None and anterior != runtime_pplx:
        raise ValueError("Cambió la configuración efectiva; no se mezclan vectores.")
    meta["modelo_runtime"] = runtime_pplx
    if not any(c["id"] == carga_pplx["id"] for c in meta["cargas"]):
        meta["cargas"].append(carga_pplx)
    print("Modelo: cuda |", runtime_pplx["dtype_pesos"], "| Carga:", round(carga_pplx["tiempo_carga_s"], 3), "s")
    print("Tokens de documentos mín./máx.:", min(runtime_pplx["tokens_por_articulo"]),
          max(runtime_pplx["tokens_por_articulo"]), "| Truncados: 0")
    print("Sin prefijos. Cuantización NATIVA de salida: int8; normalización posterior L2 en float32.")


def normalizar_salida_pplx(salida, filas):
    """No cuantiza otra vez ni reconstruye una salida flotante previa a la cuantización."""
    v = np.array(salida, dtype=np.float32, order="C", copy=True)
    if v.shape != (filas, DIMENSION) or not np.isfinite(v).all():
        raise ValueError("Forma o valores de salida incorrectos.")
    # El módulo oficial puede devolver valores cuantizados representados por floats.
    distancia_entero = float(np.max(np.abs(v - np.rint(v))))
    if v.min() < -128.00002 or v.max() > 127.00002 or distancia_entero > 2e-5:
        raise ValueError("La salida no es consistente con la cuantización int8 nativa; no se fuerza.")
    datos = {"dtype_encode": str(np.asarray(salida).dtype),
             "cuantizacion_nativa": CUANTIZACION_NATIVA,
             "min_salida_nativa": float(v.min()), "max_salida_nativa": float(v.max()),
             "desviacion_a_entero_max": distancia_entero,
             "sha256_salida_nativa_float32": hashlib.sha256(v.tobytes()).hexdigest()}
    antes = np.linalg.norm(v, axis=1)
    if not np.isfinite(antes).all() or np.any(antes <= 0):
        raise ValueError("Vector nulo o norma no finita.")
    faiss.normalize_L2(v)
    despues = np.linalg.norm(v, axis=1)
    if not np.isfinite(v).all() or not np.allclose(despues, 1, rtol=0, atol=1e-5):
        raise ValueError("Falló la normalización final.")
    datos.update({"normas_antes_min": float(antes.min()), "normas_antes_max": float(antes.max()),
                  "desviacion_despues_max": float(np.max(np.abs(despues-1)))})
    return v, datos


def codificar(textos, consulta=False, lote=LOTE):
    # Igual preparación para consultas/documentos: texto original, sin instrucciones.
    tokenizados = codificador_pplx.tokenizer(
        textos, add_special_tokens=True, padding=False, truncation=False)
    longitudes = [len(x) for x in tokenizados["input_ids"]]
    if any(n > codificador_pplx.max_seq_length for n in longitudes):
        raise ValueError("Entrada demasiado larga; no se codifica con truncamiento.")
    torch.cuda.synchronize()
    inicio = perf_counter()
    with torch.inference_mode():
        salida = codificador_pplx.encode(
            textos, batch_size=lote, show_progress_bar=False, prompt="",
            normalize_embeddings=False, precision="float32", convert_to_numpy=True,
            quantization=CUANTIZACION_NATIVA,
        )
    torch.cuda.synchronize()
    encode_s = perf_counter() - inicio
    inicio_l2 = perf_counter()
    v, datos = normalizar_salida_pplx(salida, len(textos))
    datos.update({"encode_s": encode_s, "normalizacion_s": perf_counter()-inicio_l2,
                  "tokens": longitudes, "tipo_entrada": "query" if consulta else "document",
                  "prefijo_utilizado": ""})
    return v, datos


def finalizar_indice(meta, matriz):
    if len(matriz) != len(article_ids):
        raise ValueError("No se permite crear un índice incompleto.")
    if not INDICE.exists():
        inicio = perf_counter()
        idx = faiss.IndexFlatIP(DIMENSION)
        idx.add(matriz)
        meta["tiempo_construccion_indice_s"] = perf_counter() - inicio
        # Archivo temporal para no dejar un índice final incompleto tras una interrupción.
        temporal_indice = INDICE.with_name(INDICE.name + ".tmp")
        if temporal_indice.exists():
            raise FileExistsError("Hay un índice .tmp pendiente de revisar; no se reemplaza.")
        with temporal_indice.open("xb") as f:
            f.write(faiss.serialize_index(idx).tobytes())
        if INDICE.exists():
            raise FileExistsError("El índice apareció durante la escritura; no se reemplaza.")
        temporal_indice.replace(INDICE)
    idx = faiss.read_index(str(INDICE))
    if (idx.ntotal != len(article_ids) or idx.d != DIMENSION
            or idx.metric_type != faiss.METRIC_INNER_PRODUCT
            or not np.array_equal(idx.reconstruct_n(0, len(matriz)), matriz)):
        raise ValueError("El índice existente no coincide con los vectores. No se sobrescribe.")
    meta["estado"] = "completo"
    meta["sha256_indice"] = sha(INDICE)
    guardar_estado(meta, matriz)
    cargar_estado()  # Comprobar también la relectura del NPZ.
    print("Artículos indexados:", idx.ntotal, "| Dimensión:", idx.d)
    print("Vectorización (suma de llamadas, sin piloto):", round(sum(r["encode_s"] for r in meta["lotes"]), 3), "s")
    print("Guardados:", VECTORES.name, "y", INDICE.name)
    print("SHA-256 NPZ:", sha(VECTORES))
    print("SHA-256 FAISS:", sha(INDICE))
    print("No se calcularon métricas de calidad de recuperación.")

print("Bibliotecas listas. Ningún modelo se ha cargado en esta celda.")
print("Modelo nuevo en este entorno: su ejecución real se comprueba con el piloto.")
print("Perfil fijo: pesos float32 + SDPA; salida cuantizada nativa int8. No se instala FlashAttention.")
print("La celda 3 descargará y ejecutará código oficial fijado a la misma revisión del modelo.")

W1108 00:56:15.559000 22292 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


python: 3.11.16
sistema: Windows-10-10.0.19045-SP0
numpy: 2.3.5
torch: 2.11.0+cu128
cuda_runtime: 12.8
sentence_transformers: 6.2.0
transformers: 5.22.2
huggingface_hub: 2.4.0
faiss: 1.15.1
gpu: NVIDIA GeForce RTX 4070 SUPER
VRAM libre / total (GiB): 10.81 / 11.99
Bibliotecas listas. Ningún modelo se ha cargado en esta celda.
Modelo nuevo en este entorno: su ejecución real se comprueba con el piloto.
Perfil fijo: pesos float32 + SDPA; salida cuantizada nativa int8. No se instala FlashAttention.
La celda 3 descargará y ejecutará código oficial fijado a la misma revisión del modelo.


## 3. Piloto: siete llamadas
Activa únicamente `EJECUTAR_PILOTO=True`. Se carga el modelo con su código oficial y cuantizador nativo


In [3]:
EJECUTAR_PILOTO = True
REPETIR_PILOTO = False

if not EJECUTAR_PILOTO:
    print("Preparado. Cambia EJECUTAR_PILOTO=True para la prueba pequeña, no para los 519 artículos.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if meta["estado"] == "completo" or len(matriz):
        print("Ya hay vectores del catálogo; no se repite el piloto. Consulta el registro existente.")
    elif meta["pilotos"] and meta["pilotos"][-1]["estado"] == "completo" and not REPETIR_PILOTO:
        print("Piloto ya completo; no se repite. Puedes revisar sus tiempos y continuar con la celda 4.")
    else:
        piloto = {"id": str(uuid4()), "inicio_utc": ahora(), "estado": "en_curso",
                  "entorno": ENTORNO, "sha256_consultas": SHA_CONSULTAS, "ensayos": [],
                  "revision_codigo": REVISION_CODIGO}
        meta["pilotos"].append(piloto)
        meta["estado"] = "diagnostico"
        guardar_estado(meta, matriz)
        try:
            asegurar_modelo(meta)
            guardar_estado(meta, matriz)
            longitudes = runtime_pplx["tokens_por_articulo"]
            orden = sorted(range(len(resumenes)), key=lambda i: (longitudes[i], i))
            tareas = [
                ("corto_primera_llamada", [orden[0]], False),
                ("corto_repeticion", [orden[0]], False),
                ("mediano", [orden[len(orden)//2]], False),
                ("mas_largo", [orden[-1]], False),
                ("primeros_cuatro_lote4", [0, 1, 2, 3], False),
                ("N01-ES", [], True), ("N01-EN", [], True),
            ]
            for nombre, posiciones, es_consulta in tareas:
                textos = [por_query[nombre]["consulta_propuesta"]] if es_consulta else [resumenes[i] for i in posiciones]
                print("Ejecutando:", nombre, "| Textos:", len(textos), flush=True)
                v, datos = codificar(textos, consulta=es_consulta, lote=LOTE)
                datos.update({"prueba": nombre, "fecha_utc": ahora(), "forma": list(v.shape),
                              "article_ids": [article_ids[i] for i in posiciones],
                              "consulta": textos[0] if es_consulta else None, "memoria": memoria()})
                piloto["ensayos"].append(datos)
                guardar_estado(meta, matriz)
                print("  Tiempo:", round(datos["encode_s"], 3), "s | Tokens:", datos["tokens"],
                      "| Dimensiones:", v.shape, flush=True)
                if datos["encode_s"] > PAUSA_TRAS_LLAMADA_S:
                    piloto["estado"] = "pausa_operativa"
                    print("PAUSA: llamada lenta. Conserva esta salida; no inicies el catálogo todavía.")
                    break
            else:
                piloto["estado"] = "completo"
            piloto["fin_utc"] = ahora()
            guardar_estado(meta, matriz)
            print("PILOTO:", piloto["estado"], "| Ensayos:", len(piloto["ensayos"]), "/", len(tareas))
            print("Estado de calidad: NO evaluado. Aún no hay índice del catálogo.")
        except (Exception, KeyboardInterrupt) as error:
            piloto["estado"] = "error_o_interrumpido"
            piloto["fin_utc"] = ahora()
            registrar_error(meta, matriz, "piloto", error)
            raise

Cargando PPLX desde una revisión fija; se ejecutará su código Python oficial.


config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

c:\Users\hazar\anaconda3\envs\entorno_tesis\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hazar\.cache\huggingface\hub\models--perplexity-ai--pplx-embed-v1-0.6b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


modules.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

configuration.py:   0%|          | 0.00/152 [00:00<?, ?B/s]

... (11 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

Pooling verificado: ['mean'] | include_prompt: True
Modelo: cuda | torch.float32 | Carga: 44.897 s
Tokens de documentos mín./máx.: 30 600 | Truncados: 0
Sin prefijos. Cuantización NATIVA de salida: int8; normalización posterior L2 en float32.
Ejecutando: corto_primera_llamada | Textos: 1
  Tiempo: 0.656 s | Tokens: [30] | Dimensiones: (1, 1024)
Ejecutando: corto_repeticion | Textos: 1
  Tiempo: 0.052 s | Tokens: [30] | Dimensiones: (1, 1024)
Ejecutando: mediano | Textos: 1
  Tiempo: 0.051 s | Tokens: [194] | Dimensiones: (1, 1024)
Ejecutando: mas_largo | Textos: 1
  Tiempo: 0.172 s | Tokens: [600] | Dimensiones: (1, 1024)
Ejecutando: primeros_cuatro_lote4 | Textos: 4
  Tiempo: 0.163 s | Tokens: [234, 162, 84, 180] | Dimensiones: (4, 1024)
Ejecutando: N01-ES | Textos: 1
  Tiempo: 0.047 s | Tokens: [31] | Dimensiones: (1, 1024)
Ejecutando: N01-EN | Textos: 1
  Tiempo: 0.047 s | Tokens: [18] | Dimensiones: (1, 1024)
PILOTO: completo | Ensayos: 7 / 7
Estado de calidad: NO evaluado. Aún no 

## 4. Catálogo completo, con avances por lote
Activa `EJECUTAR_CATALOGO=True` solo tras completar y revisar el piloto. 

In [4]:
EJECUTAR_CATALOGO = True

if not EJECUTAR_CATALOGO:
    print("Catálogo no iniciado. Revisa primero las siete llamadas del piloto.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if len(matriz) == len(article_ids):
        finalizar_indice(meta, matriz)  # Solo verifica o termina el guardado; no vuelve a codificar.
    else:
        if not meta["pilotos"] or meta["pilotos"][-1]["estado"] != "completo":
            raise RuntimeError("El piloto no terminó correctamente. No se procesa el catálogo todavía.")
        if meta["pilotos"][-1]["entorno"] != ENTORNO:
            raise ValueError("El entorno cambió después del piloto; revisar antes de generar el catálogo.")
        if meta["entorno_vectorizacion"] is not None and meta["entorno_vectorizacion"] != ENTORNO:
            raise ValueError("Cambió el entorno desde los avances guardados. No se mezclan ejecuciones.")
        sesion = {"id": str(uuid4()), "inicio_utc": ahora(), "filas_iniciales": len(matriz),
                  "estado": "en_curso", "entorno": ENTORNO, "revision_codigo": REVISION_CODIGO}
        meta["sesiones_catalogo"].append(sesion)
        inicio_sesion = perf_counter()
        try:
            asegurar_modelo(meta)
            meta["entorno_vectorizacion"] = ENTORNO
            meta["estado"] = "vectorizacion_incompleta"
            guardar_estado(meta, matriz)
            while len(matriz) < len(article_ids):
                comprobar_entradas()
                inicio = len(matriz)
                fin = min(inicio + LOTE, len(article_ids))
                print(f"Procesando filas {inicio+1}-{fin}/{len(article_ids)} | IDs: {article_ids[inicio:fin]}", flush=True)
                v, datos = codificar(resumenes[inicio:fin], consulta=False, lote=LOTE)
                matriz = np.concatenate([matriz, v], axis=0)
                datos.update({"inicio_fila": inicio, "fin_fila_exclusivo": fin,
                              "article_ids": article_ids[inicio:fin], "fecha_utc": ahora(),
                              "sesion_id": sesion["id"]})
                meta["lotes"].append(datos)
                guardar_estado(meta, matriz)
                print(f"  Guardadas {fin}/{len(article_ids)} | encode: {datos['encode_s']:.3f} s", flush=True)
                if datos["encode_s"] > PAUSA_TRAS_LLAMADA_S and fin < len(article_ids):
                    sesion["estado"] = "pausa_operativa"
                    meta["estado"] = "pausa_operativa"
                    print("PAUSA por lentitud; este lote ya está guardado. No se crea un índice incompleto.")
                    break
            else:
                sesion["estado"] = "terminada"
            sesion.update({"fin_utc": ahora(), "filas_finales": len(matriz),
                           "tiempo_sesion_antes_cierre_s": perf_counter()-inicio_sesion,
                           "memoria_al_fin": memoria()})
            guardar_estado(meta, matriz)
            if len(matriz) == len(article_ids):
                finalizar_indice(meta, matriz)
        except (Exception, KeyboardInterrupt) as error:
            sesion.update({"estado": "error_o_interrumpida", "fin_utc": ahora(),
                           "filas_finales": len(matriz), "tiempo_sesion_antes_cierre_s": perf_counter()-inicio_sesion})
            registrar_error(meta, matriz, "catalogo", error)
            raise

Modelo: cuda | torch.float32 | Carga: 44.897 s
Tokens de documentos mín./máx.: 30 600 | Truncados: 0
Sin prefijos. Cuantización NATIVA de salida: int8; normalización posterior L2 en float32.
Procesando filas 1-4/519 | IDs: ['1', '2', '3', '4']
  Guardadas 4/519 | encode: 0.255 s
Procesando filas 5-8/519 | IDs: ['5', '6', '7', '8']
  Guardadas 8/519 | encode: 0.195 s
Procesando filas 9-12/519 | IDs: ['9', '10', '11', '12']
  Guardadas 12/519 | encode: 0.126 s
Procesando filas 13-16/519 | IDs: ['13', '14', '15', '16']
  Guardadas 16/519 | encode: 0.266 s
Procesando filas 17-20/519 | IDs: ['17', '18', '19', '20']
  Guardadas 20/519 | encode: 0.196 s
Procesando filas 21-24/519 | IDs: ['21', '22', '23', '24']
  Guardadas 24/519 | encode: 0.150 s
Procesando filas 25-28/519 | IDs: ['25', '26', '27', '28']
  Guardadas 28/519 | encode: 0.173 s
Procesando filas 29-32/519 | IDs: ['29', '30', '31', '32']
  Guardadas 32/519 | encode: 0.310 s
Procesando filas 33-36/519 | IDs: ['33', '34', '35', '36'

## 5. Búsqueda ilustrativa, sin métricas


In [5]:
EJECUTAR_BUSQUEDA = True

if not EJECUTAR_BUSQUEDA:
    print("Búsqueda no ejecutada. Requiere NPZ completo e índice verificado.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if meta["estado"] != "completo" or len(matriz) != len(article_ids) or not INDICE.is_file():
        raise RuntimeError("Falta terminar y verificar el índice; no se busca con un NPZ parcial.")
    if meta["entorno_vectorizacion"] != ENTORNO:
        raise ValueError("El entorno de consulta cambió respecto al índice. Revisar antes de continuar.")
    idx = faiss.read_index(str(INDICE))
    if (idx.ntotal != len(article_ids) or idx.d != DIMENSION
            or idx.metric_type != faiss.METRIC_INNER_PRODUCT
            or sha(INDICE) != meta["sha256_indice"]
            or not np.array_equal(idx.reconstruct_n(0, len(matriz)), matriz)):
        raise ValueError("El índice no coincide con el NPZ.")
    try:
        asegurar_modelo(meta)
        q = por_query["N01-ES"]["consulta_propuesta"]
        vq, datos = codificar([q], consulta=True, lote=1)
        inicio = perf_counter()
        puntuaciones, posiciones = idx.search(vq, min(5, idx.ntotal))
        busqueda_s = perf_counter()-inicio
        resultados = [{"article_id": article_ids[p], "coseno": float(s), "titulo": articulos[p]["titulo"]}
                      for p, s in zip(posiciones[0].tolist(), puntuaciones[0].tolist()) if p >= 0]
        meta["busquedas"].append({"fecha_utc": ahora(), "query_id": "N01-ES", "consulta": q,
                                 "sha256_consultas": SHA_CONSULTAS,
                                 "encode_consulta": datos, "vector_consulta_float32": vq[0].tolist(),
                                 "busqueda_faiss_s": busqueda_s,
                                 "resultados": resultados, "evaluacion_pertinencia": "no_realizada", "revision_codigo": REVISION_CODIGO})
        guardar_estado(meta, matriz)
        print("Consulta:", q)
        print("Codificación de consulta:", round(datos["encode_s"], 3), "s")
        print("Búsqueda FAISS:", round(busqueda_s, 6), "s")
        print("Cinco resultados SIN calificación de pertinencia:")
        for r in resultados:
            print(f"article_id={r['article_id']} | coseno={r['coseno']:.5f} | {r['titulo'][:100]}")
        print("SHA-256 NPZ actualizado:", sha(VECTORES))
        print("SHA-256 FAISS:", sha(INDICE))
    except (Exception, KeyboardInterrupt) as error:
        registrar_error(meta, matriz, "busqueda", error)
        raise

Modelo: cuda | torch.float32 | Carga: 44.897 s
Tokens de documentos mín./máx.: 30 600 | Truncados: 0
Sin prefijos. Cuantización NATIVA de salida: int8; normalización posterior L2 en float32.
Consulta: ¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?
Codificación de consulta: 0.118 s
Búsqueda FAISS: 0.000158 s
Cinco resultados SIN calificación de pertinencia:
article_id=117 | coseno=0.58405 | A retrieval-augmented-generation pipeline to help users query system-provided documentation
article_id=184 | coseno=0.34429 | GIL-IIMAS UNAM at SemEval-2025 Task 3: MeSSI: A Multimodule System to Detect Hallucinated Segments i
article_id=273 | coseno=0.34046 | UCOM_UNAM_PLN at CheckThat! 2025: Evaluating LLMs in a two-Step Architecture for Numerical Fact Chec
article_id=175 | coseno=0.32260 | Exploring the Behavior and Performance of Large Language Models: Can LLMs Infer Answers to Question